In [1]:
import pandas as pd

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
ORIGINAL_CSV    = "webnlg_qa_dataset.csv"       # original generated questions
REGENERATED_CSV = "webnlg_qa_dataset_rebuild.csv"     # new questions from regeneration step
OUTPUT_CSV      = "webnlg_qa_merged.csv"

CSV_FIELDNAMES = [
    "split", "category", "eid",
    "lex_id", "lex_text",
    "xml_file", "mtriple", "otriple",
    "statement",
    "question_idx", "question_type", "question", "answer",
    "generation_errors",
]

# ─────────────────────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────────────────────
print("Loading CSVs…")
orig  = pd.read_csv(ORIGINAL_CSV,    dtype=str).fillna("")
regen = pd.read_csv(REGENERATED_CSV, dtype=str).fillna("")

# Keep only the CSV_FIELDNAMES columns that exist in each file
orig  = orig [[c for c in CSV_FIELDNAMES if c in orig.columns]]
regen = regen[[c for c in CSV_FIELDNAMES if c in regen.columns]]

print(f"  Original   : {len(orig):>6} rows")
print(f"  Regenerated: {len(regen):>6} rows")

# ─────────────────────────────────────────────────────────────
# CONCATENATE
# ─────────────────────────────────────────────────────────────
merged = pd.concat([orig, regen], ignore_index=True)

# ─────────────────────────────────────────────────────────────
# DEDUPLICATE
# Exact duplicate: same (eid, lex_id, question_type, question, answer)
# Keep the first occurrence (original takes priority since it's on top)
# ─────────────────────────────────────────────────────────────
dedup_keys = ["eid", "lex_id", "question_type", "question", "answer"]
before = len(merged)
merged = merged.drop_duplicates(subset=dedup_keys, keep="first")
after  = len(merged)
print(f"  Duplicates removed: {before - after}")

# Drop rows with no question (error-only placeholder rows),
# UNLESS they are the only row for that (eid, lex_id)
has_question = merged["question"].str.strip() != ""
lex_groups   = merged.groupby(["eid", "lex_id"])["question"].transform(
    lambda x: (x.str.strip() != "").any()
)
merged = merged[has_question | ~lex_groups]

# ─────────────────────────────────────────────────────────────
# REINDEX question_idx per (category, eid, lex_id)
# ─────────────────────────────────────────────────────────────
merged = merged.sort_values(
    ["category", "eid", "lex_id", "question_type", "question"],
    na_position="last"
).reset_index(drop=True)

merged["question_idx"] = (
    merged.groupby(["category", "eid", "lex_id"]).cumcount()
)

# ─────────────────────────────────────────────────────────────
# ENSURE COLUMN ORDER
# ─────────────────────────────────────────────────────────────
for col in CSV_FIELDNAMES:
    if col not in merged.columns:
        merged[col] = ""

merged = merged[CSV_FIELDNAMES]

# ─────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────
merged.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Merged CSV saved to: {OUTPUT_CSV}")
print(f"   Total rows  : {len(merged)}")
print(f"   Unique (cat, eid, lex_id): "
      f"{merged.groupby(['category','eid','lex_id']).ngroups}")

# ─────────────────────────────────────────────────────────────
# SANITY CHECK — question type distribution
# ─────────────────────────────────────────────────────────────
print("\nQuestion type distribution:")
print(merged["question_type"].value_counts().to_string())

print("\nAnswer distribution (yes_no only):")
yn = merged[merged["question_type"] == "yes_no"]
print(yn["answer"].str.strip().str.lower().value_counts().to_string())

Loading CSVs…
  Original   :  54434 rows
  Regenerated:   9276 rows
  Duplicates removed: 264

✅ Merged CSV saved to: webnlg_qa_merged.csv
   Total rows  : 63430
   Unique (cat, eid, lex_id): 8825

Question type distribution:
question_type
yes_no        43719
extractive    19710
                  1

Answer distribution (yes_no only):
answer
yes    23846
no     19873


In [3]:
import pandas as pd

TRIPLESETS_CSV = "ir_triplesets.csv"
TEXTS_CSV = "ir_texts.csv"

df_tri = pd.read_csv(TRIPLESETS_CSV)
df_txt = pd.read_csv(TEXTS_CSV)

# Normalize language column name if needed.
if "lang" not in df_tri.columns and "doc_lang" in df_tri.columns:
    df_tri = df_tri.rename(columns={"doc_lang": "lang"})

if "lang" not in df_txt.columns and "doc_lang" in df_txt.columns:
    df_txt = df_txt.rename(columns={"doc_lang": "lang"})


# =========================
# Build canonical base IDs
# =========================

def tripleset_base_id(doc_id):
    # tripleset__train__Airport__Id1__1triples__en
    # -> train__Airport__Id1__1triples
    x = str(doc_id)
    x = x.replace("tripleset__", "", 1)
    x = x.rsplit("__", 1)[0]  # remove lang
    return x

def text_base_id(doc_id):
    # text__train__Airport__Id1__1triples__en__0
    # -> train__Airport__Id1__1triples
    x = str(doc_id)
    x = x.replace("text__", "", 1)
    parts = x.split("__")
    # split, category, eid, size are first 4 chunks
    return "__".join(parts[:4])

df_tri["base_id"] = df_tri["doc_id"].apply(tripleset_base_id)
df_txt["base_id"] = df_txt["doc_id"].apply(text_base_id)

print("=" * 80)
print("RAW DOCUMENT COUNTS")
print("=" * 80)
print(f"Tripleset docs total: {len(df_tri):,}")
print(f"Text docs total:      {len(df_txt):,}")

print("\nTriplesets by language:")
display(df_tri["lang"].value_counts().sort_index().to_frame("n_docs"))

print("\nTexts by language:")
display(df_txt["lang"].value_counts().sort_index().to_frame("n_docs"))


# =========================
# Base-entry counts
# =========================

tri_base_all = set(df_tri["base_id"])
txt_base_all = set(df_txt["base_id"])

print("\n" + "=" * 80)
print("CANONICAL BASE ENTRY COVERAGE")
print("=" * 80)
print(f"Tripleset base entries: {len(tri_base_all):,}")
print(f"Text base entries:      {len(txt_base_all):,}")
print(f"Shared base entries:    {len(tri_base_all & txt_base_all):,}")
print(f"Tripleset-only entries: {len(tri_base_all - txt_base_all):,}")
print(f"Text-only entries:      {len(txt_base_all - tri_base_all):,}")


# =========================
# Language-specific base-entry coverage
# =========================

langs = sorted(set(df_tri["lang"].dropna()) | set(df_txt["lang"].dropna()))

rows = []

for lang in langs:
    tri_ids = set(df_tri.loc[df_tri["lang"] == lang, "base_id"])
    txt_ids = set(df_txt.loc[df_txt["lang"] == lang, "base_id"])

    rows.append({
        "lang": lang,
        "tripleset_base_entries": len(tri_ids),
        "text_base_entries": len(txt_ids),
        "shared": len(tri_ids & txt_ids),
        "tripleset_only": len(tri_ids - txt_ids),
        "text_only": len(txt_ids - tri_ids),
    })

coverage_by_lang = pd.DataFrame(rows)
print("\nLanguage-specific tripleset/text base coverage:")
display(coverage_by_lang)


# =========================
# Find mismatches
# =========================

mismatch_rows = []

for lang in langs:
    tri_ids = set(df_tri.loc[df_tri["lang"] == lang, "base_id"])
    txt_ids = set(df_txt.loc[df_txt["lang"] == lang, "base_id"])

    for bid in sorted(tri_ids - txt_ids):
        mismatch_rows.append({
            "lang": lang,
            "base_id": bid,
            "status": "tripleset_without_text",
        })

    for bid in sorted(txt_ids - tri_ids):
        mismatch_rows.append({
            "lang": lang,
            "base_id": bid,
            "status": "text_without_tripleset",
        })

mismatches = pd.DataFrame(mismatch_rows)

print("\nMismatches:")
print(f"Total mismatches: {len(mismatches):,}")

if len(mismatches):
    display(mismatches.head(100))
    mismatches.to_csv("triplesets_texts_mismatches_by_lang.csv", index=False)
    print("Saved: triplesets_texts_mismatches_by_lang.csv")
else:
    print("No language-specific mismatches between triplesets and texts.")


# =========================
# Size distribution comparison
# =========================

print("\n" + "=" * 80)
print("SIZE DISTRIBUTION")
print("=" * 80)

tri_size = (
    df_tri
    .groupby(["lang", "size"])
    .size()
    .rename("tripleset_docs")
    .reset_index()
)

txt_size = (
    df_txt
    .groupby(["lang", "size"])
    .size()
    .rename("text_docs")
    .reset_index()
)

size_cmp = tri_size.merge(
    txt_size,
    on=["lang", "size"],
    how="outer",
).fillna(0)

size_cmp["tripleset_docs"] = size_cmp["tripleset_docs"].astype(int)
size_cmp["text_docs"] = size_cmp["text_docs"].astype(int)
size_cmp["diff_text_minus_tripleset"] = (
    size_cmp["text_docs"] - size_cmp["tripleset_docs"]
)

display(size_cmp.sort_values(["lang", "size"]))


# =========================
# Check strict 1:1 per language
# =========================

strict_ok = (
    (coverage_by_lang["tripleset_only"] == 0).all()
    and (coverage_by_lang["text_only"] == 0).all()
)

print("\n" + "=" * 80)
print("STRICT 1:1 CHECK")
print("=" * 80)

if strict_ok:
    print("Every tripleset base entry has a matching text base entry for each language.")
else:
    print("There are language-specific tripleset/text mismatches. Inspect mismatches above.")

RAW DOCUMENT COUNTS
Tripleset docs total: 33,262
Text docs total:      33,286

Triplesets by language:


,n_docs
lang,
en,16631
es,16631



Texts by language:


,n_docs
lang,
en,16643
es,16643



CANONICAL BASE ENTRY COVERAGE
Tripleset base entries: 16,631
Text base entries:      16,631
Shared base entries:    16,631
Tripleset-only entries: 0
Text-only entries:      0

Language-specific tripleset/text base coverage:


,lang,tripleset_base_entries,text_base_entries,shared,tripleset_only,text_only
0,en,16631,16631,16631,0,0
1,es,16631,16631,16631,0,0



Mismatches:
Total mismatches: 0
No language-specific mismatches between triplesets and texts.

SIZE DISTRIBUTION


,lang,size,tripleset_docs,text_docs,diff_text_minus_tripleset
0,en,1,3967,3967,0
1,en,2,3134,3146,12
2,en,3,3405,3405,0
3,en,4,3174,3174,0
4,en,5,2330,2330,0
5,en,6,342,342,0
6,en,7,279,279,0
7,es,1,3967,3967,0
8,es,2,3134,3146,12
9,es,3,3405,3405,0



STRICT 1:1 CHECK
Every tripleset base entry has a matching text base entry for each language.


In [2]:
import os
import glob
import xml.etree.ElementTree as ET

WEBNLG_ROOT = "./WebNLG_ES"

missing_keys = set(
    missing_es_df[["split", "category", "eid", "size", "xml_file"]]
    .astype(str)
    .apply(tuple, axis=1)
)

rows = []

for split in ["train", "dev", "test"]:
    xml_files = glob.glob(os.path.join(WEBNLG_ROOT, split, "**", "*.xml"), recursive=True)

    for xml_path in xml_files:
        xml_file = os.path.basename(xml_path)

        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
        except ET.ParseError:
            continue

        for entry in root.iter("entry"):
            category = entry.get("category", "")
            eid = entry.get("eid", "")
            size = entry.get("size", "")

            key = (split, category, eid, size, xml_file)

            if key not in missing_keys:
                continue

            mtriples = [
                x.text.strip()
                for x in entry.iter("mtriple")
                if x.text and x.text.strip()
            ]

            striples = [
                x.text.strip()
                for x in entry.iter("striple")
                if x.text and x.text.strip()
            ]

            has_spanishtripleset_tag = entry.find("spanishtripleset") is not None

            rows.append({
                "split": split,
                "category": category,
                "eid": eid,
                "size": size,
                "xml_file": xml_file,
                "has_spanishtripleset_tag": has_spanishtripleset_tag,
                "n_mtriples": len(mtriples),
                "n_striples": len(striples),
                "mtriples": " ||| ".join(mtriples),
                "striples": " ||| ".join(striples),
            })

diagnostic_missing_es = pd.DataFrame(rows)
display(diagnostic_missing_es)

diagnostic_missing_es.to_csv("diagnostic_missing_spanish_triplesets.csv", index=False)
print("Saved: diagnostic_missing_spanish_triplesets.csv")

""


Saved: diagnostic_missing_spanish_triplesets.csv
